# Proof Techniques, from Zero — with SageMath 10.9
### assuming you have never written a proof

Every example is taken from the Ising partition function you are already computing, so nothing here
is a toy problem. By the end you will have proved, properly, the statement your other notebook only
*checked*:

$$\operatorname{Tr}(T^N)=\lambda_+^N+\lambda_-^N$$

**Books on your shelf, used throughout**

| | where | use |
|---|---|---|
| Devlin, *Introduction to Mathematical Thinking* (117 pp) | `Foundation_Math_Books_z` | §3.1–3.5. Read alongside |
| Diedrichs & Lovett, *Transition to Advanced Mathematics* (552 pp) | `Foundation_Math_Books_z` | Ch. 1–2 for drills |
| Axler, *Linear Algebra Done Right* | `Z_Ref_Books/01_…` | the theorems we cite |

---
## §0. What a proof actually is

A **proof** is a finite chain of statements, each of which is either

* something already agreed (a definition, an axiom, or a previously proved theorem), or
* something that follows from earlier statements by a rule of logic,

ending at the thing you claimed.

That is the whole definition. A proof is not a calculation, not evidence, and not a demonstration
that something works in examples. **It is an argument that leaves no case unexamined.**

> **The distinction this notebook exists to teach.** Your other notebook printed
> `Tr(T^ns) == lam_+^ns + lam_-^ns -> True` for `ns = 3, 5, 8`.
> That is *three facts*. The theorem is a claim about **every** $N$ — infinitely many. No amount of
> checking gets you there. Something else must.

## §1. Setup

In [1]:
print(version())
# Reminder: never use N as a variable in Sage - N() is the numerical evaluation function.
beta = var('beta', domain='real')
print('ready')

SageMath version 10.9, Release Date: 2026-05-04


ready


---
## §2. The logic you need first

Almost every theorem has the shape **"if $p$ then $q$"**, written $p\Rightarrow q$.

Two things beginners get wrong, both worth fixing now.

**(a) $p\Rightarrow q$ is only false when $p$ is true and $q$ is false.** In particular, if $p$ is
false the implication is *true* regardless of $q$. "If the moon is cheese then $1=2$" is a true
statement.

**(b) $p\Rightarrow q$ does not mean $q\Rightarrow p$.** "Symmetric $\Rightarrow$ diagonalisable" is
true. The reverse is false, and in §5 you will kill it with a counterexample.

In [2]:
print('  p       q      p => q')
print('  ' + '-'*24)
for p in (True, False):
    for q in (True, False):
        implication = (not p) or q          # this IS the definition of =>
        print(f'  {str(p):6s}  {str(q):6s} {str(implication):6s}')
print()
print('Note rows 3 and 4: when p is False, p => q is True either way.')

  p       q      p => q
  ------------------------
  True    True   True  
  True    False  False 
  False   True   True  
  False   False  True  

Note rows 3 and 4: when p is False, p => q is True either way.


### Quantifiers, and how to negate them

| symbol | reads | to **disprove** it you must |
|---|---|---|
| $\forall x,\;P(x)$ | "for all $x$, $P(x)$" | exhibit **one** $x$ with $P(x)$ false — a **counterexample** |
| $\exists x,\;P(x)$ | "there exists $x$ with $P(x)$" | show **every** $x$ fails — much harder |

$$\neg\big(\forall x\,P(x)\big)\;\equiv\;\exists x\,\neg P(x)$$

**This asymmetry is the most useful fact in elementary logic.** Disproving a "for all" is cheap.
Proving one is expensive. Always try to disprove first — §5.

*(Devlin §2.4; Diedrichs & Lovett §1.5–1.6)*

---
## §3. Technique 1 — Direct proof

**Shape.** Assume $p$. Reason forward. Arrive at $q$.

> **Theorem.** The Ising transfer matrix $T$ is symmetric.

**Proof.** By definition $T(s,s')=\exp\!\big[\beta J s s' + \tfrac{\beta h}{2}(s+s')\big]$.
Swapping $s$ and $s'$ leaves $ss'$ unchanged (multiplication commutes) and leaves $s+s'$ unchanged
(addition commutes). So $T(s,s')=T(s',s)$, which is what "symmetric" means. $\blacksquare$

Three things to notice: we **used the definition**, we **used known facts** (commutativity), and we
**stopped when we reached the claim**. That is a complete proof, and it is four lines.

In [3]:
J, h = var('J h', domain='real')
def T_entry(s, sp): return exp(beta*J*s*sp + beta*h*(s+sp)/2)
T = matrix(SR, 2, 2, [[T_entry(s,sp) for sp in (1,-1)] for s in (1,-1)]).simplify_full()
show(T)
print('Sage agrees T is symmetric:', T.is_symmetric())
print()
print('But note: Sage CHECKED a 2x2 matrix. The proof above explains WHY,')
print('and would work for any number of spin states, not just two.')

[e^(J*beta + beta*h)         e^(-J*beta)]
[        e^(-J*beta) e^(J*beta - beta*h)]

Sage agrees T is symmetric: True

But note: Sage CHECKED a 2x2 matrix. The proof above explains WHY,
and would work for any number of spin states, not just two.


---
## §4. Technique 2 — Proof by cases

**Shape.** Split all possibilities into finitely many cases that cover everything. Prove each.

> **Theorem.** For every real $\beta$, $\;\lambda_+ \ge \lambda_-\;$ where $\lambda_\pm = 2\cosh\beta,\;2\sinh\beta$.

**Proof.** $\lambda_+-\lambda_-=2\cosh\beta-2\sinh\beta=2e^{-\beta}$.

* **Case 1, $\beta \ge 0$:** then $e^{-\beta}>0$, so the difference is positive.
* **Case 2, $\beta<0$:** then $-\beta>0$ so $e^{-\beta}>1>0$, still positive.

Both cases give a positive difference, and every real $\beta$ falls in one of them. $\blacksquare$

*(Actually $e^{-\beta}>0$ for all real $\beta$, so the cases were unnecessary — a good lesson: look
for the one-line argument before splitting.)*

In [4]:
gap = (2*cosh(beta) - 2*sinh(beta)).exponentialize().simplify_full()
print('lambda_+ - lambda_- simplifies to:', gap)
print('which is 2*e^(-beta), positive for every real beta.')
print()
for b in [-3, -1, 0, 1, 3]:
    print(f'   beta={b:3d}   difference = {N(gap.subs(beta==b)):10.6f}')

lambda_+ - lambda_- simplifies to: 2*e^(-beta)
which is 2*e^(-beta), positive for every real beta.

   beta= -3   difference =  40.171074
   beta= -1   difference =   5.436564
   beta=  0   difference =   2.000000
   beta=  1   difference =   0.735759
   beta=  3   difference =   0.099574


---
## §5. Technique 3 — Counterexample (how to *disprove*)

Cheapest technique in mathematics. To kill $\forall x\,P(x)$, produce **one** $x$ where $P$ fails.

### Your turn — three claims. Two are false.

1. Every diagonalisable matrix has **orthogonal** eigenvectors.
2. Every real symmetric matrix has **distinct** eigenvalues.
3. Every real symmetric matrix is diagonalisable.

Think before running the cell.

In [5]:
print('CLAIM 1: every diagonalisable matrix has orthogonal eigenvectors')
A = matrix(QQ, 2, 2, [1,1,0,2])
D, P = A.eigenmatrix_right()
v0, v1 = P.column(0), P.column(1)
print('   A =', A.list(), '  diagonalisable?', A.is_diagonalizable(), '  symmetric?', A.is_symmetric())
print('   eigenvectors:', v0, v1, '  dot product =', v0.dot_product(v1))
print('   => FALSE. Dot product is 1, not 0.')
print()
print('CLAIM 2: every real symmetric matrix has distinct eigenvalues')
I2 = identity_matrix(QQ, 2)
print('   I =', I2.list(), '  symmetric?', I2.is_symmetric(), '  eigenvalues:', I2.eigenvalues())
print('   => FALSE. Both eigenvalues are 1.')
print()
print('CLAIM 3: every real symmetric matrix is diagonalisable')
print('   => TRUE, and no counterexample exists. This is Axler 7.29,')
print('      the Real Spectral Theorem. It needs a PROOF, not a search.')

CLAIM 1: every diagonalisable matrix has orthogonal eigenvectors


   A = [1, 1, 0, 2]   diagonalisable? True   symmetric? False
   eigenvectors: (1, 1) (1, 0)   dot product = 1
   => FALSE. Dot product is 1, not 0.

CLAIM 2: every real symmetric matrix has distinct eigenvalues
   I = [1, 0, 0, 1]   symmetric? True   eigenvalues: [1, 1]
   => FALSE. Both eigenvalues are 1.

CLAIM 3: every real symmetric matrix is diagonalisable
   => TRUE, and no counterexample exists. This is Axler 7.29,
      the Real Spectral Theorem. It needs a PROOF, not a search.


### The lesson

Claims 1 and 2 died to a single $2\times2$ matrix each. Claim 3 survived every search you could
run — but **surviving a search is not a proof.** Axler spends six pages proving 7.29.

This is exactly the asymmetry from §2: *disproof is a search, proof is an argument.*

> **Why claim 1 matters to you.** It is precisely why Axler 7.29 says **orthonormal** and not merely
> "a basis of eigenvectors". Orthogonality is extra, and it is what makes the trace argument in §8
> work cleanly.

---
## §6. Technique 4 — Contrapositive

$p\Rightarrow q$ is *logically identical* to $\neg q\Rightarrow\neg p$. Sometimes the second is
easier.

> **Theorem.** If $T$ has two non-orthogonal eigenvectors, then $T$ is not real symmetric.

Proving that directly is awkward. The contrapositive is "if $T$ is real symmetric then all its
eigenvectors (for distinct eigenvalues) are orthogonal" — which is Axler 7.29 again. Same statement,
easier door.

In [6]:
print('The two columns of the truth table are identical:')
print('   p      q      p=>q    ¬q=>¬p')
for p in (True, False):
    for q in (True, False):
        print(f'   {str(p):6s} {str(q):6s} {str((not p) or q):7s} {str(q or (not p)):6s}')
print()
print('Identical in all four rows => the two forms are interchangeable.')

The two columns of the truth table are identical:
   p      q      p=>q    ¬q=>¬p
   True   True   True    True  
   True   False  False   False 
   False  True   True    True  
   False  False  True    True  

Identical in all four rows => the two forms are interchangeable.


---
## §7. Technique 5 — Contradiction

**Shape.** Assume the *opposite* of what you want. Derive something impossible. Conclude the
assumption was wrong.

> **Theorem.** The 1D Ising chain has no phase transition at any finite $\beta>0$.

**Proof.** A phase transition means the free energy $f=-\frac1\beta\ln(2\cosh\beta)$ fails to be
analytic at some finite $\beta_c>0$.

Suppose it does. The only way $\ln(2\cosh\beta)$ can fail to be analytic is if its argument hits the
branch point, i.e. $2\cosh\beta_c=0$. But $\cosh\beta=\tfrac12(e^\beta+e^{-\beta})$ is a sum of two
strictly positive numbers, so $\cosh\beta_c>0$ — it can never be zero.

Contradiction. So no such $\beta_c$ exists. $\blacksquare$

In [7]:
print('solve(2*cosh(beta) == 0, beta)  ->', solve(2*cosh(beta) == 0, beta))
print('   (empty list: no real solution, exactly as the proof argued)')
print()
f = -log(2*cosh(beta))/beta
print('f =', f)
print('and f is smooth - all derivatives finite at beta = 1:')
g = f
for k in range(1,5):
    g = diff(g, beta)
    print(f'   d^{k}f/dbeta^{k} = {N(g.subs(beta==1)):+.6f}')

solve(2*cosh(beta) == 0, beta)  -> [

]
   (empty list: no real solution, exactly as the proof argued)

f = -log(2*cosh(beta))/beta
and f is smooth - all derivatives finite at beta = 1:
   d^1f/dbeta^1 = +0.365334
   d^2f/dbeta^2 = -1.150642
   d^3f/dbeta^3 = +4.091626
   d^4f/dbeta^4 = -16.988131


---
## §8. Technique 6 — Induction, and the drill

**The big one.** Induction proves $\forall n\in\mathbb{N}$ statements — infinitely many claims — with
finite work.

**Shape.** To prove $P(n)$ for all $n\ge1$:
1. **Base case.** Prove $P(1)$.
2. **Inductive step.** Assume $P(k)$ for an arbitrary $k$; prove $P(k+1)$ follows.

Then $P(n)$ holds for all $n$: $P(1)$ gives $P(2)$, which gives $P(3)$, forever.

### The lemma your other notebook assumed

> **Lemma.** If $D$ is diagonal with entries $d_1,d_2$, then $D^n$ is diagonal with entries $d_1^n,d_2^n$.

**Proof by induction on $n$.**

*Base case $n=1$.* $D^1=D$, diagonal with entries $d_1^1,d_2^1$. ✓

*Inductive step.* Assume $D^k=\operatorname{diag}(d_1^k,d_2^k)$. Then
$$D^{k+1}=D^k\cdot D=\operatorname{diag}(d_1^k,d_2^k)\cdot\operatorname{diag}(d_1,d_2)
=\operatorname{diag}(d_1^{k}d_1,\;d_2^{k}d_2)=\operatorname{diag}(d_1^{k+1},d_2^{k+1}),$$
using only that the product of two diagonal matrices multiplies entrywise. ✓

Both steps hold, so the lemma holds for every $n\ge1$. $\blacksquare$

In [8]:
D = diagonal_matrix(SR, [2*cosh(beta), 2*sinh(beta)])
print('Sage checks n = 1..20:')
print('   all diagonal?', all((D^n).is_diagonal() for n in range(1,21)))
print('   D^3 diagonal entries:', (D^3).diagonal())
print()
print('*** READ THIS CAREFULLY ***')
print('Sage verified 20 cases. The lemma claims INFINITELY many.')
print('The 20 checks are evidence that we did not misstate it.')
print('The PROOF is the inductive step above - and only that.')

Sage checks n = 1..20:


   all diagonal? True
   D^3 diagonal entries: [8*cosh(beta)^3, 8*sinh(beta)^3]

*** READ THIS CAREFULLY ***
Sage verified 20 cases. The lemma claims INFINITELY many.
The 20 checks are evidence that we did not misstate it.
The PROOF is the inductive step above - and only that.


---
## §9. Putting it together — the theorem, proved

> **Theorem.** For the 1D Ising transfer matrix $T$ and every $N\ge1$,
> $$\operatorname{Tr}(T^N)=\lambda_+^N+\lambda_-^N.$$

**Proof.**

1. $T$ is real and symmetric — **§3, direct proof**.
2. Hence by the **Real Spectral Theorem (Axler 7.29)** there is an orthonormal basis of eigenvectors.
   Write $P$ for the matrix of those eigenvectors and $D=\operatorname{diag}(\lambda_+,\lambda_-)$,
   so that $P^{-1}TP=D$.
3. Then $P^{-1}T^NP=(P^{-1}TP)^N=D^N$, and by the **Lemma of §8 (induction)** $D^N$ is diagonal with
   entries $\lambda_\pm^N$.
4. The trace is **invariant under change of basis** (**Axler 10.A**), so
   $\operatorname{Tr}(T^N)=\operatorname{Tr}(D^N)=\lambda_+^N+\lambda_-^N$. $\blacksquare$

Four steps, each licensed by something named. **That is a proof.** It holds for $N=10^{100}$ without
any further work.

In [9]:
T0 = T.subs(h==0, J==1)
D_, P_ = T0.eigenmatrix_right()      # NOTE: Sage returns (D, P), diagonal FIRST
lam = [e.simplify_full() for e in T0.eigenvalues()]
lam_p = max(lam, key=lambda e: N(e.subs(beta==1)))
lam_m = min(lam, key=lambda e: N(e.subs(beta==1)))

def proves_zero(expr):
    return bool(expr.exponentialize().simplify_full() == 0)

print('step 2 -- P^-1 T P is diagonal:', (P_.inverse()*T0*P_).simplify_full().is_diagonal())
print('step 2 -- eigenvectors orthogonal:', P_.column(0).dot_product(P_.column(1)) == 0)
print()
print('step 4 -- verify a few instances (NOT the proof):')
for ns in (1, 2, 3, 7, 12):
    lhs = (T0^ns).trace().simplify_full()
    rhs = (lam_p^ns + lam_m^ns).simplify_full()
    print(f'   ns={ns:2d}:  {proves_zero(lhs - rhs)}')

step 2 -- P^-1 T P is diagonal: True
step 2 -- eigenvectors orthogonal: 0 == 0

step 4 -- verify a few instances (NOT the proof):
   ns= 1:  True
   ns= 2:  True
   ns= 3:  True
   ns= 7:  True
   ns=12:  True


---
## §10. Where the CAS will lie to you

Sage returning `False` does **not** mean a statement is false. It means Sage could not *show* it
true with the simplification you asked for. This bit you already met the hard way.

In [10]:
expr = 2*cosh(beta) - (exp(2*beta)+1)*exp(-beta)     # this is identically ZERO
print('the expression:', expr)
print()
print('  bool(expr == 0)                       ->', bool(expr == 0))
print('  bool(expr.simplify_full() == 0)       ->', bool(expr.simplify_full() == 0))
print('  bool(expr.canonicalize_radical()==0)  ->', bool(expr.canonicalize_radical() == 0))
print('  bool(expr.exponentialize()...  == 0)  ->', bool(expr.exponentialize().simplify_full() == 0))
print()
print('Three methods say False. One says True. The expression IS zero -')
print('substitute any number and see:')
for b in [0.3, 1.0, 2.5]:
    print(f'   beta={b}:  {N(expr.subs(beta==b)):.3e}')
print()
print('MORAL: a CAS "False" is not a disproof. It is a failure to simplify.')
print('A CAS "True" from a sound method IS a verification - but only of what you asked.')

the expression: -(e^(2*beta) + 1)*e^(-beta) + 2*cosh(beta)

  bool(expr == 0)                       -> False
  bool(expr.simplify_full() == 0)       -> False
  bool(expr.canonicalize_radical()==0)  -> False
  bool(expr.exponentialize()...  == 0)  -> True

Three methods say False. One says True. The expression IS zero -
substitute any number and see:
   beta=0.300000000000000:  0.000e-12
   beta=1.00000000000000:  -4.441e-16
   beta=2.50000000000000:  -1.776e-15

MORAL: a CAS "False" is not a disproof. It is a failure to simplify.
A CAS "True" from a sound method IS a verification - but only of what you asked.


---
## §11. Pure $\to$ applied $\to$ computational: one theorem, three languages

| | statement | lives in |
|---|---|---|
| **Pure** | A self-adjoint operator on a finite-dimensional real inner-product space has an orthonormal eigenbasis | Axler 7.29 |
| **Applied** | The free energy is $-\frac1\beta\ln\lambda_+$, and $\xi=1/\ln(\lambda_+/\lambda_-)$ | Feynman ch. 5; PBoC2 ch. 6 |
| **Computational** | `T0.eigenmatrix_right()` returns $(D,P)$ with $P$ orthogonal | Sage |

Same mathematics. The pure statement is what *licenses* the applied one; the computational one only
*checks instances*. Losing track of which you are doing is the commonest error in computational
science — it is the whole argument of the Part VI dossier.

---
## §12. Exercises

**Logic** *(Devlin §2; D&L §1.5)*
1. Write $\neg(\forall\beta>0,\;\lambda_+(\beta)>\lambda_-(\beta))$ using $\exists$. What would you have to produce to prove it?
2. Is "if $T$ is diagonalisable then $T$ is symmetric" true? Prove or give a counterexample.

**Direct & cases** *(D&L §2.2–2.3)*
3. Prove $\lambda_+\lambda_- = \det T$ directly from $T$'s entries, with $h=0,J=1$.
4. Prove $\lambda_->0$ for $\beta>0$ but $\lambda_-<0$ for $\beta<0$. Use cases.

**Induction** *(D&L §4.5)*
5. Prove $\operatorname{Tr}(A^n)=\sum_i\lambda_i^n$ for **any** diagonalisable $n\times n$ matrix. Where is symmetry used, and where is it not?
6. Prove: if $P$ is invertible then $(P^{-1}AP)^n=P^{-1}A^nP$. *(This is step 3 of §9 — we used it without proof.)*

**Contradiction**
7. Prove $\xi=1/\ln\coth\beta$ is finite for every $\beta>0$, by assuming it is infinite.

**CAS discipline**
8. Find another identity `simplify_full()` cannot close but `exponentialize()` can.
9. Take exercise 5's statement and have Sage check it for five random $4\times4$ symmetric matrices. Then explain, in one sentence, why that is not a proof.